# Orders table: cleaning

Orders table: cleaning code compiled from the notebook, in the order it was run.
Cells are separated by "# %%" so VS Code can run this file cell by cell,
or you can paste each block into its own notebook cell.

Run from the folder that contains raw/ (your VS Code workspace root).

Decisions and findings for this table are written up in `cleaning_notes/README_orders.md`.

### 1. Imports and settings

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

RAW = Path('raw')
CLEAN = Path('cleaned')

### 2. Load the raw orders file

In [2]:
orders = pd.read_csv(RAW / 'olist_orders_dataset.csv')

### 3. First inspection: structure, nulls, status categories

In [3]:
orders.info()
print(orders.isna().sum())
print(orders['order_status'].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_deliv

### 4. Convert the five timestamp columns from text to datetime

In [4]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
]
orders[date_cols] = orders[date_cols].apply(pd.to_datetime)
orders.info()   # all five should now show datetime64[us], with the same non-null counts

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


### 5. Duplicate check

In [5]:
# order_id is the key, so if it is unique, no two rows can be identical.
print(orders['order_id'].is_unique)                 # expect True
print(orders['order_id'].nunique() == len(orders))  # expect True

True
True


### 6. Nulls in context: separate legitimate nulls from contradictions

In [6]:
delivered = orders['order_status'] == 'delivered'
no_cust_date = orders['order_delivered_customer_date'].isna()

# Nulls among delivered orders only
print(orders.loc[delivered].isna().sum())   # approved_at 14, carrier 2, customer 8

# Reverse check: non-delivered orders that have a customer delivery date
print(orders.loc[~delivered, 'order_delivered_customer_date'].notna().sum())   # expect 6

order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64
6


### 7. Flag orders whose delivery date contradicts their status (date_has_issue)

In [7]:
not_delivered = ~delivered
has_cust_date = orders['order_delivered_customer_date'].notna()

orders['date_has_issue'] = (delivered & no_cust_date) | (not_delivered & has_cust_date)
print(orders['date_has_issue'].value_counts())   # expect False 99,427 / True 14

date_has_issue
False    99427
True        14
Name: count, dtype: int64


### 8. Delivery features (whole days; nulls stay null)

In [8]:
orders['delivery_time_days'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.days

orders['delivery_delay_days'] = (
    orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']
).dt.days

# is_late: True / False where a delay exists, null (not False) where it does not.
# A plain "> 0" comparison would turn missing delays into False.
orders['is_late'] = (
    (orders['delivery_delay_days'] > 0)
    .astype('boolean')
    .where(orders['delivery_delay_days'].notna())
)

# Nullable integers, so the columns display as 8 instead of 8.0
orders['delivery_time_days'] = orders['delivery_time_days'].astype('Int64')
orders['delivery_delay_days'] = orders['delivery_delay_days'].astype('Int64')

### 9. Verify the new columns

In [9]:
print(orders['is_late'].value_counts(dropna=False))   # False 89,941 / True 6,535 / <NA> 2,965
print(orders[['delivery_time_days', 'delivery_delay_days']].describe())

is_late
False    89941
True      6535
<NA>      2965
Name: count, dtype: Int64
       delivery_time_days  delivery_delay_days
count             96476.0              96476.0
mean            12.094086           -11.876881
std              9.551746            10.183854
min                   0.0               -147.0
25%                   6.0                -17.0
50%                  10.0                -12.0
75%                  15.0                 -7.0
max                 209.0                188.0


### 10. Timestamp sequence checks (counts of rows breaking the expected order)

In [10]:
print((orders['order_delivered_carrier_date'] < orders['order_purchase_timestamp']).sum())   # 166
print((orders['order_approved_at'] > orders['order_delivered_carrier_date']).sum())          # 1,359
print((orders['order_delivered_customer_date'] < orders['order_delivered_carrier_date']).sum())  # 23
print((orders['order_delivered_customer_date'] < orders['order_purchase_timestamp']).sum())  # 0
print((orders['order_approved_at'] < orders['order_purchase_timestamp']).sum())              # 0

166
1359
23
0
0


### 11. Named masks and the combined sequence flag

In [11]:
mask_a = orders['order_delivered_carrier_date'] < orders['order_purchase_timestamp']
mask_b = orders['order_approved_at'] > orders['order_delivered_carrier_date']
mask_c = orders['order_delivered_customer_date'] < orders['order_delivered_carrier_date']

# Why 166 + 1,359 + 23 gives 1,382 and not 1,548:
print((mask_a & ~mask_b).sum())   # 0: every mask_a row is also in mask_b
print((mask_b & mask_c).sum())    # 0: b and c do not overlap

orders['has_sequence_issue'] = mask_a | mask_b | mask_c
print(orders['has_sequence_issue'].value_counts())   # False 98,059 / True 1,382

# How the two flags relate (they do not overlap)
print(pd.crosstab(orders['date_has_issue'], orders['has_sequence_issue']))

0
0
has_sequence_issue
False    98059
True      1382
Name: count, dtype: int64
has_sequence_issue  False  True 
date_has_issue                  
False               98045   1382
True                   14      0


### 12. Combined flag for dashboards

In [12]:
orders['has_data_issue'] = orders['date_has_issue'] | orders['has_sequence_issue']
print(orders['has_data_issue'].value_counts())   # False 98,045 / True 1,396

has_data_issue
False    98045
True      1396
Name: count, dtype: int64


### 13. Save the cleaned table

In [13]:
CLEAN.mkdir(exist_ok=True)
orders.to_csv(CLEAN / 'orders_clean.csv', index=False)
print(orders.shape)   # expect (99441, 14)

(99441, 14)
